# TakeMeter: Anime Discourse Classifier

This notebook fine-tunes `distilbert-base-uncased` to classify r/anime posts into three discourse types:
- **analysis**: Structured arguments with evidence
- **hot_take**: Bold opinions with little evidence
- **reaction**: Immediate emotional responses

It also runs a zero-shot baseline using Groq's `llama-3.3-70b-versatile` for comparison.

## 1. Setup & Installation

In [ ]:
!pip install -q transformers datasets scikit-learn groq accelerate

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from groq import Groq
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Load Data

In [ ]:
from google.colab import files

print("Upload labeled_dataset.csv:")
uploaded = files.upload()

In [ ]:
df = pd.read_csv("labeled_dataset.csv")
print(f"Total examples: {len(df)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nLabel percentages:")
print(df["label"].value_counts(normalize=True).round(3) * 100)

In [ ]:
LABELS = sorted(df["label"].unique().tolist())
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

print(f"Labels: {LABELS}")
print(f"Label to ID: {LABEL2ID}")

df["label_id"] = df["label"].map(LABEL2ID)

## 3. Split Data (70% Train / 15% Validation / 15% Test)

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
)

print(f"Train: {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"Test: {len(test_df)} examples")
print(f"\nTrain label distribution:")
print(train_df["label"].value_counts())
print(f"\nTest label distribution:")
print(test_df["label"].value_counts())

## 4. Zero-Shot Baseline (Groq LLaMA 3.3 70B)

In [ ]:
import getpass

GROQ_API_KEY = getpass.getpass("Enter your Groq API key: ")

In [ ]:
client = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """You are classifying posts from the r/anime community on Reddit.

Labels:
analysis = A structured argument supported by specific evidence, examples from shows, comparison, or reasoning about anime themes, characters, animation, or storytelling.
hot_take = A bold opinion stated confidently with little or no supporting evidence, often about rankings, quality judgments, or controversial comparisons.
reaction = An immediate emotional response to an episode, scene, announcement, or personal viewing experience, with little argument or structured reasoning.

Classify the post below.
Return ONLY one label name (analysis, hot_take, or reaction). No explanation."""


def classify_with_groq(text, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Post:\n{text}"},
                ],
                temperature=0,
                max_tokens=10,
            )
            result = response.choices[0].message.content.strip().lower()
            result = result.replace("'", "").replace('"', "").strip()
            if result in LABELS:
                return result
            for label in LABELS:
                if label in result:
                    return label
            return result
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"Error classifying: {e}")
                return "error"


print(f"Classifying {len(test_df)} test examples with Groq...")
baseline_preds = []
for i, row in test_df.iterrows():
    pred = classify_with_groq(row["text"])
    baseline_preds.append(pred)
    time.sleep(0.5)

test_df = test_df.copy()
test_df["baseline_pred"] = baseline_preds
print("Done!")

In [ ]:
valid_baseline = test_df[test_df["baseline_pred"].isin(LABELS)].copy()
unparseable = test_df[~test_df["baseline_pred"].isin(LABELS)]

print(f"Parseable predictions: {len(valid_baseline)}/{len(test_df)}")
if len(unparseable) > 0:
    print(f"\nUnparseable outputs:")
    for _, row in unparseable.iterrows():
        print(f"  Predicted: '{row['baseline_pred']}' | Text: {row['text'][:80]}...")

baseline_accuracy = accuracy_score(valid_baseline["label"], valid_baseline["baseline_pred"])
print(f"\nBaseline Accuracy: {baseline_accuracy:.4f}")
print(f"\nBaseline Classification Report:")
baseline_report = classification_report(
    valid_baseline["label"],
    valid_baseline["baseline_pred"],
    labels=LABELS,
    output_dict=True,
)
print(
    classification_report(
        valid_baseline["label"], valid_baseline["baseline_pred"], labels=LABELS
    )
)

baseline_cm = confusion_matrix(
    valid_baseline["label"], valid_baseline["baseline_pred"], labels=LABELS
)
print(f"\nBaseline Confusion Matrix:")
print(baseline_cm)

## 5. Fine-Tune DistilBERT

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=256
    )


train_dataset = Dataset.from_pandas(
    train_df[["text", "label_id"]].rename(columns={"label_id": "labels"})
)
val_dataset = Dataset.from_pandas(
    val_df[["text", "label_id"]].rename(columns={"label_id": "labels"})
)
test_dataset = Dataset.from_pandas(
    test_df[["text", "label_id"]].rename(columns={"label_id": "labels"})
)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")
print(f"Test dataset: {len(test_dataset)} examples")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)
print(f"Model loaded on {device}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
print("Training complete!")

## 6. Evaluate Fine-Tuned Model

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
pred_label_names = [ID2LABEL[p] for p in pred_labels]
true_label_names = [ID2LABEL[t] for t in predictions.label_ids]

ft_accuracy = accuracy_score(true_label_names, pred_label_names)
print(f"Fine-Tuned Model Accuracy: {ft_accuracy:.4f}")
print(f"\nClassification Report:")
ft_report = classification_report(
    true_label_names, pred_label_names, labels=LABELS, output_dict=True
)
print(classification_report(true_label_names, pred_label_names, labels=LABELS))

In [ ]:
ft_cm = confusion_matrix(true_label_names, pred_label_names, labels=LABELS)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=ft_cm, display_labels=LABELS)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Fine-Tuned DistilBERT — Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix.png")

## 7. Comparison: Baseline vs Fine-Tuned

In [ ]:
print("=" * 60)
print("BASELINE vs FINE-TUNED COMPARISON")
print("=" * 60)
print(f"\nOverall Accuracy:")
print(f"  Baseline (Groq LLaMA 3.3 70B): {baseline_accuracy:.4f}")
print(f"  Fine-Tuned (DistilBERT):        {ft_accuracy:.4f}")
print(f"  Improvement:                    {ft_accuracy - baseline_accuracy:+.4f}")

print(f"\nPer-Class F1 Scores:")
print(f"{'Label':<12} {'Baseline':>10} {'Fine-Tuned':>12} {'Diff':>8}")
print("-" * 44)
for label in LABELS:
    b_f1 = baseline_report.get(label, {}).get("f1-score", 0)
    f_f1 = ft_report.get(label, {}).get("f1-score", 0)
    diff = f_f1 - b_f1
    print(f"{label:<12} {b_f1:>10.4f} {f_f1:>12.4f} {diff:>+8.4f}")

## 8. Export Results

In [ ]:
results = {
    "model": "distilbert-base-uncased",
    "community": "r/anime",
    "labels": LABELS,
    "dataset_size": len(df),
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "training_args": {
        "epochs": 3,
        "learning_rate": 2e-5,
        "batch_size": 16,
    },
    "baseline": {
        "model": "llama-3.3-70b-versatile",
        "provider": "Groq",
        "accuracy": round(baseline_accuracy, 4),
        "per_class": {
            label: {
                "precision": round(baseline_report[label]["precision"], 4),
                "recall": round(baseline_report[label]["recall"], 4),
                "f1": round(baseline_report[label]["f1-score"], 4),
            }
            for label in LABELS
            if label in baseline_report
        },
    },
    "fine_tuned": {
        "accuracy": round(ft_accuracy, 4),
        "per_class": {
            label: {
                "precision": round(ft_report[label]["precision"], 4),
                "recall": round(ft_report[label]["recall"], 4),
                "f1": round(ft_report[label]["f1-score"], 4),
            }
            for label in LABELS
            if label in ft_report
        },
    },
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved evaluation_results.json")
print(json.dumps(results, indent=2))

## 9. Sample Classifications with Confidence

In [ ]:
probs = torch.nn.functional.softmax(
    torch.tensor(predictions.predictions), dim=-1
)
confidences = probs.max(dim=-1).values.numpy()

sample_df = test_df.copy().reset_index(drop=True)
sample_df["predicted"] = pred_label_names
sample_df["confidence"] = confidences
sample_df["correct"] = sample_df["label"] == sample_df["predicted"]

print("\n" + "=" * 80)
print("SAMPLE CLASSIFICATIONS")
print("=" * 80)

samples = pd.concat([
    sample_df[sample_df["correct"]].nlargest(3, "confidence"),
    sample_df[~sample_df["correct"]].head(2),
]).head(5)

for _, row in samples.iterrows():
    status = "CORRECT" if row["correct"] else "WRONG"
    print(f"\n[{status}] Confidence: {row['confidence']:.4f}")
    print(f"  True: {row['label']} | Predicted: {row['predicted']}")
    print(f"  Text: {row['text'][:120]}...")
    print("-" * 80)

## 10. Error Analysis

In [ ]:
errors = sample_df[~sample_df["correct"]].copy()
print(f"Total errors: {len(errors)}/{len(sample_df)}")
print(f"\nError breakdown:")

for _, row in errors.iterrows():
    print(f"\n{'=' * 80}")
    print(f"True: {row['label']} -> Predicted: {row['predicted']} (conf: {row['confidence']:.4f})")
    print(f"Text: {row['text'][:200]}")

print(f"\n\nConfusion patterns in errors:")
if len(errors) > 0:
    error_patterns = errors.groupby(["label", "predicted"]).size().reset_index(name="count")
    error_patterns = error_patterns.sort_values("count", ascending=False)
    for _, row in error_patterns.iterrows():
        print(f"  {row['label']} misclassified as {row['predicted']}: {row['count']} times")

## 11. Download Results

Run this cell to download the evaluation files to your local machine.

In [ ]:
from google.colab import files

files.download("evaluation_results.json")
files.download("confusion_matrix.png")